# Day 20 · Ontology, vectors or both? A grounding ablation

**First-principles question:** Does the agent need to search for meaning, be told what Acme's words mean, or both?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import json
import numpy as np
import pandas as pd
from lab.acme import DATA, INTENTS, load_cases, save_results
from lab.evals import score
from lab.version_b import version_b

cases = load_cases()
by_id = {c["id"]: c for c in cases}
ONTOLOGY = json.loads((DATA / "ontology.json").read_text(encoding="utf-8"))
CONTEXTS = ["oracle", "retrieved", "ontology", "hybrid"]

## 1 · Is the ontology complete?
Every intent needs a meaning, synonyms (include Hinglish) and an offer envelope.

In [ ]:
for name in INTENTS:
    entry = ONTOLOGY["intents"].get(name, {})
    print(f"{name:11} synonyms {len(entry.get('synonyms', [])):2}  offers {len(entry.get('allowed_offers', [])):2}  never {len(entry.get('never', [])):2}")

## 2 · Four contexts on every case
First add `context="ontology"` and `context="hybrid"` to `lab/version_b.py`.

In [ ]:
runs = {}
for ctx in CONTEXTS:
    runs[ctx] = [dict(version_b(c, context=ctx), id=c["id"]) for c in cases]
    save_results(f"B_{ctx}", runs[ctx])
passed = {ctx: np.array([score(by_id[r["id"]], r)["passed"] for r in rows], dtype=float) for ctx, rows in runs.items()}

In [ ]:
def row(ctx):
    p, n = passed[ctx].mean(), len(passed[ctx])
    # TODO: return {"context": ctx, "pass rate": p, "± 2 se": 2 * standard error of p}
    ...

def paired(a, b):
    """Mean of b − a over the same cases, and ± 2 standard errors."""
    # TODO: d = b - a; return d.mean(), 2 * d.std(ddof=1) / sqrt(len(d))
    ...

display(pd.DataFrame([row(c) for c in CONTEXTS]).set_index("context").round(3))
m, r = paired(passed["retrieved"], passed["hybrid"])
print(f"hybrid − retrieved = {m:+.1%} ± {r:.1%}")

**Which grounding would you recommend to Acme, and what would keeping the ontology current cost?** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
for name in INTENTS:
    entry = ONTOLOGY["intents"][name]
    assert entry["means"] and entry["synonyms"] and entry["allowed_offers"], f"ontology.json: fill in {name}"
assert all(len(runs[c]) == len(cases) for c in CONTEXTS), "Step 2: run every context on every case"
assert row("oracle")["± 2 se"] > 0 or passed["oracle"].mean() in (0, 1)
m, r = paired(np.array([1, 0, 0, 1.0]), np.array([1, 1, 0, 1.0]))
assert abs(m - 0.25) < 1e-9 and abs(r - 0.5) < 1e-9, "paired() should give +25% ± 50% on the example"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")